In [103]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [104]:
import os

In [105]:
path_proj_datasets = "/content/drive/MyDrive/llm_from_scratch/datasets"

In [106]:
os.chdir(path_proj_datasets)
print(os.getcwd())

/content/drive/MyDrive/llm_from_scratch/datasets


In [107]:
import pandas as pd

In [108]:
df = pd.read_csv("/content/drive/MyDrive/llm_from_scratch/datasets/SMSSpamCollection", sep="\t")

In [109]:
df.head()

,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."
0,ham,Ok lar... Joking wif u oni...
1,spam,Free entry in 2 a wkly comp to win FA Cup fina...
2,ham,U dun say so early hor... U c already then say...
3,ham,"Nah I don't think he goes to usf, he lives aro..."
4,spam,FreeMsg Hey there darling it's been 3 week's n...


In [110]:
df.head(0)

,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat..."


In [111]:
df.columns = ['l', 't']

In [112]:
df.rename(columns={'l': 'Label', 't': 'Text'}, inplace=True)

In [113]:
df.head()

,Label,Text
0,ham,Ok lar... Joking wif u oni...
1,spam,Free entry in 2 a wkly comp to win FA Cup fina...
2,ham,U dun say so early hor... U c already then say...
3,ham,"Nah I don't think he goes to usf, he lives aro..."
4,spam,FreeMsg Hey there darling it's been 3 week's n...


In [114]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5571 entries, 0 to 5570
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Label   5571 non-null   object
 1   Text    5571 non-null   object
dtypes: object(2)
memory usage: 87.2+ KB


In [115]:
df.describe()

,Label,Text
count,5571,5571
unique,2,5168
top,ham,"Sorry, I'll call later"
freq,4824,30


In [116]:
print((df['Label'] == 'spam').sum())
print((df['Label'] == 'ham').sum())

747
4824


In [117]:
df.shape

(5571, 2)

In [118]:
def create_balanced_dataset(df):
    spam_df = df[df['Label'] == 'spam'].copy()
    ham_df = df[df['Label'] == 'ham'].copy()

    min_len = min(len(spam_df), len(ham_df))

    spam_balanced = spam_df.sample(min_len, random_state=42)
    ham_balanced = ham_df.sample(min_len, random_state=42)

    balanced_df = pd.concat([spam_balanced, ham_balanced]) \
                   .sample(frac=1, random_state=42) \
                   .reset_index(drop=True)

    return balanced_df

In [119]:
df = create_balanced_dataset(df)

In [120]:
df.head()

,Label,Text
0,ham,"Come to mu, we're sorting out our narcotics si..."
1,ham,Oh yah... We never cancel leh... Haha
2,ham,I'm stuck in da middle of da row on da right h...
3,ham,Gokila is talking with you aha:)
4,ham,We confirm eating at esplanade?


In [136]:
df.shape

(1494, 2)

In [122]:
len(df[df['Label'] == 'spam'])

747

In [123]:
len(df[df['Label'] == 'ham'])

747

In [124]:
df['Label'] = df['Label'].map({'spam': 1, 'ham': 0})

In [125]:
df.head()

,Label,Text
0,0,"Come to mu, we're sorting out our narcotics si..."
1,0,Oh yah... We never cancel leh... Haha
2,0,I'm stuck in da middle of da row on da right h...
3,0,Gokila is talking with you aha:)
4,0,We confirm eating at esplanade?


In [126]:
def split_dataset(df, train_ratio, validation_ratio):
    train_end = int(len(df) * train_ratio)
    validation_end = train_end + (int(len(df) * validation_ratio))

    # now spliting dataset into train, validation and test:
    train_df = df[:train_end]
    validation_df = df[train_end:validation_end]
    test_df = df[validation_end:]
    return train_df, validation_df, test_df

In [127]:
train_df, validation_df, test_df = split_dataset(df, 0.6, 0.2)

In [128]:
print(train_df.shape, test_df.shape, validation_df.shape)
print("Original df shape:" , df.shape[0])
print("splitted df sum shape:", train_df.shape[0] + test_df.shape[0] + validation_df.shape[0])

(896, 2) (300, 2) (298, 2)
Original df shape: 1494
splitted df sum shape: 1494


In [129]:
train_df.to_csv("train_df.csv", index=None)
validation_df.to_csv("validation_df.csv", index=None)
test_df.to_csv("test_df.csv", index=None)

In [130]:
# dataloader class:
import torch
from torch.utils.data import Dataset, DataLoader

In [131]:
class SpamDataset(Dataset):
    def __init__(self, csv_df, tokenizer, max_length=None, pad_token_id=50256):
        self.df = pd.read_csv(csv_df)
        self.encoded_text = [
            tokenizer.encode(text) for text in self.df['Text']
        ]

        if max_length is None:
            self.max_length = max([len(text) for text in self.encoded_text])
        else:
            self.max_length = max_length

            self.encoded_text = [
                text[:self.max_length] for text in self.encoded_text
                ]

        self.encoded_text = [
            text + [pad_token_id] * (self.max_length - len(text)) for text in self.encoded_text
        ]


    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):
        encoded = self.encoded_text[idx]
        label = self.df['Label'][idx]
        encoded = torch.tensor(encoded, dtype=torch.long)
        label = torch.tensor(label, dtype=torch.long)
        return encoded, label

In [132]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')
train_df = '/content/drive/MyDrive/llm_from_scratch/datasets/train_df.csv'
validation_df = '/content/drive/MyDrive/llm_from_scratch/datasets/validation_df.csv'
test_df = '/content/drive/MyDrive/llm_from_scratch/datasets/test_df.csv'

In [133]:
train_dataset = SpamDataset(train_df, tokenizer)
validation_dataset = SpamDataset(validation_df, tokenizer)
test_dataset = SpamDataset(test_df, tokenizer)

In [134]:
batch_size = 8
train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
validation_dataloader = DataLoader(validation_dataset, batch_size=batch_size, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)

In [135]:
input_batch, target_batch = next(iter(train_dataloader))
input_batch.shape, target_batch.shape

(torch.Size([8, 169]), torch.Size([8]))